In [1]:
import lsdb
import numpy as np
import matplotlib.pyplot as plt
from dask.distributed import Client
from nested_pandas import read_parquet
from lsdb.core.search.region_search import MOCSearch

/astro/users/midai/.conda/envs/lf_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
lsdb.__version__

'0.10.4'

In [3]:
import nested_pandas
nested_pandas.__version__

'0.6.10'

In [4]:
# client = Client(n_workers=1, memory_limit="10 GiB", threads_per_worker=1)
# display(client)
client = Client(n_workers=20, memory_limit="24 GiB", threads_per_worker=1)
display(client)

/astro/users/midai/.conda/envs/lf_env/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34829 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:34829/status,
Dashboard: http://127.0.0.1:34829/status,Workers: 20
Total threads: 20,Total memory: 480.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38633,Workers: 0
Dashboard: http://127.0.0.1:34829/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:43365,Total threads: 1
Dashboard: http://127.0.0.1:42369/status,Memory: 24.00 GiB
Nanny: tcp://127.0.0.1:43563,


2026-09-06 00:22:32,041 - distributed.scheduler - WARNING - Worker failed to heartbeat for 1366s; attempting restart: <WorkerState 'tcp://127.0.0.1:33201', name: 9, status: running, memory: 0, processing: 0>
2026-09-06 00:22:34,373 - distributed.scheduler - WARNING - Worker failed to heartbeat for 1366s; attempting restart: <WorkerState 'tcp://127.0.0.1:33251', name: 11, status: running, memory: 0, processing: 0>
2026-09-06 00:22:34,462 - distributed.scheduler - WARNING - Worker failed to heartbeat for 1366s; attempting restart: <WorkerState 'tcp://127.0.0.1:33429', name: 14, status: running, memory: 0, processing: 0>
2026-09-06 00:22:34,462 - distributed.scheduler - WARNING - Worker failed to heartbeat for 1366s; attempting restart: <WorkerState 'tcp://127.0.0.1:35429', name: 5, status: running, memory: 0, processing: 0>
2026-09-06 00:22:34,463 - distributed.scheduler - WARNING - Worker failed to heartbeat for 1366s; attempting restart: <WorkerState 'tcp://127.0.0.1:36921', name: 7, s

In [5]:
dia_object_path = '/astro/store/shire/hats/dash/hats/dp2_rc/dia_object_collection'
object_path = '/astro/store/shire/hats/dash/hats/dp2_rc/object_collection'

In [6]:
dia_object = lsdb.open_catalog(dia_object_path,
                               columns=["diaObjectId","diaObjectForcedSource.band"]
                               )

In [7]:
def compute_nobs(df):
    if len(df) == 0:
        df = df.assign(**{name: np.array([], dtype=np.int32) for name in ["nobs"]})
    else:
        df["nobs"] = df["diaObjectForcedSource"].len()
    return df

In [8]:
for f in "grizuy":
    band_dia_object = dia_object.query(f"diaObjectForcedSource.band == '{f}'")
    band_dia_object = band_dia_object.map_partitions(compute_nobs)
    band_dia_object = band_dia_object.query("nobs > 0")
    band_dia_object.write_catalog(f"outputs/{f}_dia_object",overwrite=True)

/astro/users/midai/.conda/envs/lf_env/lib/python3.11/site-packages/distributed/client.py:3415: UserWarning: Sending large graph of size 12.28 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
Writing Catalog: 100%|██████████| 8393/8393 [11:59<00:00, 11.66it/s]  
/astro/users/midai/.conda/envs/lf_env/lib/python3.11/site-packages/distributed/client.py:3415: UserWarning: Sending large graph of size 12.30 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
Writing Margin Cache: 100%|██████████| 8388/8388 [01:40<00:00, 83